##### First neural network (Iris Dataset)
- Uses numpy
- Here Softmax and ReLU is used
    - Softmax
        - Softmax is used at the output
        - It predicts the probaility of each output neuron (0.7 Cats, 0.3 Dogs)
    - ReLU
        - Relu lives in the hidden layer and used mainly in Backpropogation
        - It processes the features and decides which "signals" are important enough to pass forward. (What flavour coffee gets through)
- Cross-Entropy insted of MSE
    - Cross-Entropy measure distance between 2 probability distributions. Unlike MSE, which measures the distance between 2 points
    - In classification tasks, your model outputs a probability (e.g., 80% Cat, 20% Dog). Cross-Entropy calculates how much this prediction deviates from the true label (100% Cat, 0% Dog).
- If you use MSE with a Sigmoid or Softmax output, the gradients become very flat when the model is "wrong." This leads back to the Vanishing Gradient Problem
- Sigmoid is **not** used because of the **Vanishing Gradient Problem**. Sigmois is majorly is used for **Binary Classification Taks**, while Softmax and ReLU is used in **Multi-Class Classification Task**

In [1]:
import numpy as np
from sklearn.datasets import load_iris
from sklearn.preprocessing import StandardScaler, OneHotEncoder


- iris dataset
- y.reshape() - used because to convert 1D row to 2D column
- Y has output as [0,1,2] which are the classification, but can be considered as numbers by the code. To prevent that, OneHotEncoder is used

In [2]:

# 1. Setup Data
iris = load_iris()
X = iris.data
y = iris.target.reshape(-1, 1)

# Preprocessing: Scale features and One-Hot Encode labels
scaler = StandardScaler()
X = scaler.fit_transform(X)
encoder = OneHotEncoder(sparse_output=False)
y_onehot = encoder.fit_transform(y)

# 2. Initialize Weights & Biases
np.random.seed(42)
input_sz, hidden_sz, output_sz = 4, 8, 3


- *0.01 is used to keep the weights non-zero. It is technically not necessery but is a good practice

In [3]:

W1 = np.random.randn(input_sz, hidden_sz) * 0.01
b1 = np.zeros((1, hidden_sz))
W2 = np.random.randn(hidden_sz, output_sz) * 0.01
b2 = np.zeros((1, output_sz))

learning_rate = 0.05


In [4]:

# 3. Training Loop
for epoch in range(1000):
    # --- FORWARD PASS ---
    # Hidden Layer (ReLU)
    z1 = np.dot(X, W1) + b1
    a1 = np.maximum(0, z1) 
    
    # Output Layer (Softmax)
    z2 = np.dot(a1, W2) + b2
    exp_z2 = np.exp(z2 - np.max(z2, axis=1, keepdims=True)) # Stability trick
    softmax_out = exp_z2 / np.sum(exp_z2, axis=1, keepdims=True)
    
    # --- BACKPROPAGATION ---
    # Output layer error (Softmax + Cross-Entropy derivative is simple: pred - target)
    dz2 = (softmax_out - y_onehot) / X.shape[0]
    dW2 = np.dot(a1.T, dz2)
    db2 = np.sum(dz2, axis=0, keepdims=True)
    
    # Hidden layer error (Backprop through ReLU)
    da1 = np.dot(dz2, W2.T)
    dz1 = da1 * (z1 > 0) # Gradient of ReLU
    dW1 = np.dot(X.T, dz1)
    db1 = np.sum(dz1, axis=0, keepdims=True)
    
    # --- UPDATE WEIGHTS ---
    W2 -= learning_rate * dW2
    b2 -= learning_rate * db2
    W1 -= learning_rate * dW1
    b1 -= learning_rate * db1

    if epoch % 100 == 0:
        loss = -np.mean(np.sum(y_onehot * np.log(softmax_out + 1e-9), axis=1))
        print(f"Epoch {epoch}, Loss: {loss:.4f}")

# 4. Final Accuracy
final_preds = np.argmax(softmax_out, axis=1)
accuracy = np.mean(final_preds == iris.target)
print(f"\nFinal Training Accuracy: {accuracy * 100:.2f}%")

Epoch 0, Loss: 1.0987
Epoch 100, Loss: 1.0873
Epoch 200, Loss: 0.6075
Epoch 300, Loss: 0.3841
Epoch 400, Loss: 0.2900
Epoch 500, Loss: 0.2299
Epoch 600, Loss: 0.1856
Epoch 700, Loss: 0.1523
Epoch 800, Loss: 0.1281
Epoch 900, Loss: 0.1107

Final Training Accuracy: 97.33%


##### Using PyTorch and nn.Module

In [6]:
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler


In [9]:
iris = load_iris()
X = iris.data          # shape (150, 4)
y = iris.target        # labels 0,1,2

# Train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Standardize features (important for MLPs)
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# Convert to torch tensors
X_train = torch.tensor(X_train, dtype=torch.float32)
y_train = torch.tensor(y_train, dtype=torch.long)
X_test = torch.tensor(X_test, dtype=torch.float32)
y_test = torch.tensor(y_test, dtype=torch.long)


In [11]:
class IrisNet(nn.Module):
    def __init__(self, input_dim=4, hidden_dim=16, output_dim=3):
        super().__init__()
        self.fc1 = nn.Linear(input_dim, hidden_dim)
        self.fc2 = nn.Linear(hidden_dim, output_dim)
        self.relu = nn.ReLU()

    def forward(self, x):
        x = self.fc1(x)
        x = self.relu(x)
        x = self.fc2(x)   # raw logits
        return x

model = IrisNet()
print(model)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.01)


IrisNet(
  (fc1): Linear(in_features=4, out_features=16, bias=True)
  (fc2): Linear(in_features=16, out_features=3, bias=True)
  (relu): ReLU()
)


In [12]:
num_epochs = 100

for epoch in range(num_epochs):
    model.train()
    # Forward
    outputs = model(X_train)
    loss = criterion(outputs, y_train)

    # Backward + update
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    if (epoch + 1) % 10 == 0:
        # Compute train accuracy
        with torch.no_grad():
            preds = outputs.argmax(dim=1)
            acc = (preds == y_train).float().mean().item()
        print(f"Epoch {epoch+1}/{num_epochs}, "
              f"Loss: {loss.item():.4f}, "
              f"Train Acc: {acc:.3f}")


Epoch 10/100, Loss: 0.7467, Train Acc: 0.850
Epoch 20/100, Loss: 0.4837, Train Acc: 0.833
Epoch 30/100, Loss: 0.3533, Train Acc: 0.875
Epoch 40/100, Loss: 0.2634, Train Acc: 0.925
Epoch 50/100, Loss: 0.2095, Train Acc: 0.933
Epoch 60/100, Loss: 0.1625, Train Acc: 0.942
Epoch 70/100, Loss: 0.1268, Train Acc: 0.967
Epoch 80/100, Loss: 0.1015, Train Acc: 0.967
Epoch 90/100, Loss: 0.0844, Train Acc: 0.967
Epoch 100/100, Loss: 0.0733, Train Acc: 0.975


In [13]:
model.eval()
with torch.no_grad():
    test_outputs = model(X_test)
    test_preds = test_outputs.argmax(dim=1)
    test_acc = (test_preds == y_test).float().mean().item()

print(f"Test accuracy: {test_acc:.3f}")


Test accuracy: 0.933


##### Scalable depth and Width

- The above code have a model as class based and the following has the model as method based.
- class Based
    - it uses nn.Module
    - Since classes are a clean slate, forward module is defines because the code will not know the flow of code.
    - Hence, the forward method is described
- method based
    - it uses nn.Sequential(*layers)
    - Sequential is a inbuilt fn that contains forward method.
    - the statements are executed in the the order as they are defined.
    - **It is important to define the statements in order so that it is executed in order**

In [14]:
def make_iris_mlp(input_dim=4, hidden_dims=[16], output_dim=3):
    layers = []
    prev_dim = input_dim
    for h in hidden_dims:
        layers.append(nn.Linear(prev_dim, h))
        layers.append(nn.ReLU())
        prev_dim = h
    layers.append(nn.Linear(prev_dim, output_dim))
    return nn.Sequential(*layers)

def train_and_eval(hidden_dims, lr=0.01, epochs=100):
    model = make_iris_mlp(hidden_dims=hidden_dims)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=lr)

    for epoch in range(epochs):
        model.train()
        outputs = model(X_train)
        loss = criterion(outputs, y_train)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

    model.eval()
    with torch.no_grad():
        test_outputs = model(X_test)
        test_preds = test_outputs.argmax(dim=1)
        test_acc = (test_preds == y_test).float().mean().item()
    print(f"Hidden dims {hidden_dims}: test acc = {test_acc:.3f}")
    return test_acc

configs = [
    [8],         # 1 hidden layer, width 8
    [16],        # 1 hidden layer, width 16
    [32],        # 1 hidden layer, width 32
    [16, 16],    # 2 hidden layers
    [32, 16],    # 2 hidden layers, decreasing
]

results = {}
for cfg in configs:
    acc = train_and_eval(cfg)
    results[tuple(cfg)] = acc

print(results)


Hidden dims [8]: test acc = 0.967
Hidden dims [16]: test acc = 0.967
Hidden dims [32]: test acc = 0.967
Hidden dims [16, 16]: test acc = 0.967
Hidden dims [32, 16]: test acc = 0.967
{(8,): 0.9666666388511658, (16,): 0.9666666388511658, (32,): 0.9666666388511658, (16, 16): 0.9666666388511658, (32, 16): 0.9666666388511658}
